# Подбор аугментаций для исходного датасета
**Датасет**: teeth-seg-3537 Computer Vision Model      
**Автор**: Godento2 ([Roboflow](https://universe.roboflow.com/godento2/teeth-seg-3537-iaky1))



## Импорт библиотек

In [1]:
import os
from google.colab import drive
from tqdm.notebook import tqdm
import zipfile
import cv2
import numpy as np
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import random
from pathlib import Path

import matplotlib.pyplot as plt
# настройка визуализации
%matplotlib inline
plt.style.use('seaborn-v0_8')
%config InlineBackend.figure_format = "retina"

## Установка параметров

In [2]:
DATASET_PATH_ZIP = '/content/gdrive/MyDrive/Colab_Notebooks/Startup/02_Datasets/Godento2v2.zip'
DATASET_PATH_UNZIP = '/content'
DATASET_DIR = '/content/dataset'

## Получение датасета

In [3]:
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [4]:
# Распаковываем
with zipfile.ZipFile(DATASET_PATH_ZIP, 'r') as zip_ref:
    for file in tqdm(zip_ref.namelist(), desc='Распаковка архива'):
        zip_ref.extract(file, DATASET_PATH_UNZIP)

# переименовываем каталог с распакованным датасетом
os.rename('/content/Godento2v2', '/content/dataset')

print(f"\nАрхив успешно распакован в: {DATASET_DIR}")

Распаковка архива:   0%|          | 0/11560 [00:00<?, ?it/s]


Архив успешно распакован в: /content/dataset


## Визуализация выбранных трансформаций

In [5]:
IMG_SIZE = 640  # Размер изображения

In [6]:
# Функция для определения трансформаций для обучения нашей модели
def get_train_transforms(img_size=IMG_SIZE):
    """
    Аугментации для ортопантомограмм
    Специфично для сегментации зубов на рентгеновских снимках.
    """
    return A.Compose([
        # ------------------------------------------------------------
        # 1. ГЕОМЕТРИЧЕСКИЕ ПРЕОБРАЗОВАНИЯ
        # ------------------------------------------------------------
        A.Affine(
            scale=(0.95, 1.05),               # Масштаб: случайное увеличение/уменьшение до ±5%
            translate_percent=(0.03, 0.03),   # Сдвиг по горизонтали/вертикали до 3% от размера
            rotate=(-3, 3),                   # Поворот до ±3 градусов
            p=0.5,                            # Вероятность применения 50%
            border_mode=cv2.BORDER_CONSTANT,  # Новые области заполняются константой (чёрным)
            fill=0,                           # Значение для заполнения на изображении (0 — чёрный)
            fill_mask=0                       # Значение для заполнения на маске (0 — фон)
        ),
        # ОПИСАНИЕ: лёгкие аффинные искажения имитируют небольшие повороты головы пациента,
        # разный масштаб съёмки и смещения челюсти в кадре. Все изменения незначительны,
        # чтобы не нарушить анатомическую структуру зубного ряда.

        # ------------------------------------------------------------
        # 2. МЯГКИЕ ЭЛАСТИЧНЫЕ ДЕФОРМАЦИИ (применяются редко)
        # ------------------------------------------------------------
        A.ElasticTransform(
            alpha=0.5,                            # Интенсивность сдвига пикселей (малое значение → слабая деформация)
            sigma=25,                             # Степень размытия поля деформации (сглаживает искажения)
            p=0.1,                                # Вероятность 10% — очень осторожное применение
            mask_interpolation=cv2.INTER_NEAREST  # Для масок — интерполяция без сглаживания
        ),
        # ОПИСАНИЕ: ElasticTransform моделирует незначительные неоднородности тканей или
        # естественные искривления челюсти. Параметры подобраны так, чтобы зубы оставались
        # узнаваемыми, но контуры слегка «дышали». Высокая вероятность не используется,
        # чтобы не создавать неестественных форм.

        # ------------------------------------------------------------
        # 3. КОРРЕКЦИЯ КОНТРАСТА И ЯРКОСТИ (РЕНТГЕН-СПЕЦИФИЧНЫЕ)
        # ------------------------------------------------------------
        A.CLAHE(
            clip_limit=2.0,             # Порог ограничения гистограммы (выше — сильнее контраст)
            tile_grid_size=(8, 8),      # Размер ячеек для локального выравнивания на изображении (ОПТГ)
            p=0.5
        ),
        # ОПИСАНИЕ: CLAHE (Contrast Limited Adaptive Histogram Equalization) —
        # стандартный метод для рентгеновских снимков. Улучшает локальный контраст,
        # делая более чёткими границы зубов и корней, особенно на затемнённых участках.
        # Параметры (8,8) и clip_limit=2.0 дают умеренное усиление.

        A.RandomBrightnessContrast(
            brightness_limit=0.08,      # Изменение яркости до ±8%
            contrast_limit=0.08,        # Изменение контраста до ±8%
            p=0.5
        ),
        # Описание: Имитирует вариации экспозиции при съёмке — разные аппараты,
        # настройки яркости, толщина мягких тканей.

        # ------------------------------------------------------------
        # 4. ИМИТАЦИЯ АРТЕФАКТОВ (CoarseDropout) (применяются редко)
        # ------------------------------------------------------------
        A.CoarseDropout(
            num_holes_range=(1, 2),         # Количество прямоугольных артефактов: 1 или 2
            hole_height_range=(0.02, 0.04), # Высота артефакта 2–8% от высоты изображения
            hole_width_range=(0.02, 0.04),  # Ширина артефакта 2–8% от ширины
            fill=0,                         # Заливка артефакта чёрным (0) на изображении
            fill_mask=0,                    # Заливка артефакта фоном (0) на маске
            p=0.05                          # Вероятность 5% — редкое событие
        ),
        A.CoarseDropout(
            num_holes_range=(1, 2),
            hole_height_range=(0.02, 0.04),
            hole_width_range=(0.02, 0.04),
            fill=128,                       # Заливка артефакта серым (128) на изображении
            fill_mask=0,
            p=0.05
        ),
        A.CoarseDropout(
            num_holes_range=(1, 2),
            hole_height_range=(0.02, 0.04),
            hole_width_range=(0.02, 0.04),
            fill=255,                       # Заливка артефакта белым (255) на изображении
            fill_mask=0,
            p=0.05
        ),
        # ОПИСАНИЕ: CoarseDropout заменяет случайные прямоугольные области на указанный цвет.
        # В контексте ортопантомограмм это имитирует:
        #   - артефакты движения (смазанные участки);
        #   - наложение посторонних предметов (зажимы, маркеры);
        #   - отсутствие части зуба (например, разрушенная коронка);
        #   - переэкспонированные участки, пломбы, коронки, металлические вкладки.
        # Маленькая вероятность и небольшие размеры артефактов (2–8%) предотвращают
        # чрезмерное зашумление датасета и позволяют модели научиться игнорировать
        # локальные дефекты.

        # ------------------------------------------------------------
        # 5. ДОБАВЛЕНИЕ ШУМА (GaussNoise)
        # ------------------------------------------------------------
        A.GaussNoise(std_range=(0.01, 0.04), p=0.2),
        # ОПИСАНИЕ: Добавляет гауссов шум, имитируя зернистость рентгеновской
        # плёнки или электронные шумы. Параметры std_range по умолчанию (10.0, 50.0)
        # дают умеренный уровень шума, достаточный для повышения робастности модели,
        # но не разрушающий мелкие детали (например, периодонтальную щель).

        # ------------------------------------------------------------
        # 6. НОРМАЛИЗАЦИЯ И ПРЕОБРАЗОВАНИЕ В ТЕНЗОР PyTorch
        # ------------------------------------------------------------
        A.Normalize(mean=(0.5,), std=(0.5,)),
        ToTensorV2()
    ])

In [7]:
def get_individual_transforms(img_size=IMG_SIZE):
    """Возвращает словарь с отдельными трансформациями для визуализации"""
    transforms_dict = {
        'Affine (масштаб, поворот, сдвиг)': A.Compose([
            A.Affine(
                scale=(0.95, 1.05),
                translate_percent=(0.03, 0.03),
                rotate=(-3, 3),
                p=1.0,
                border_mode=cv2.BORDER_CONSTANT,
                fill=0,
                fill_mask=0
            )
        ]),

        'ElasticTransform (деформация)': A.Compose([
            A.ElasticTransform(
                alpha=0.5,
                sigma=25,
                p=1.0,
                mask_interpolation=cv2.INTER_NEAREST
            )
        ]),

        'CLAHE (контраст)': A.Compose([
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=1.0)
        ]),

        'Brightness & Contrast': A.Compose([
            A.RandomBrightnessContrast(
                brightness_limit=0.08,
                contrast_limit=0.08,
                p=1.0
            )
        ]),

        'CoarseDropout (черные артефакты)': A.Compose([
            A.CoarseDropout(
                num_holes_range=(1, 2),
                hole_height_range=(0.02, 0.08),
                hole_width_range=(0.02, 0.08),
                fill=0,
                fill_mask=0,
                p=1.0
            )
        ]),

        'CoarseDropout (серые артефакты)': A.Compose([
            A.CoarseDropout(
                num_holes_range=(1, 2),
                hole_height_range=(0.02, 0.08),
                hole_width_range=(0.02, 0.08),
                fill=128,
                fill_mask=0,
                p=1.0
            )
        ]),

        'CoarseDropout (белые артефакты)': A.Compose([
            A.CoarseDropout(
                num_holes_range=(1, 2),
                hole_height_range=(0.02, 0.08),
                hole_width_range=(0.02, 0.08),
                fill=255,
                fill_mask=0,
                p=1.0
            )
        ]),

        'GaussNoise (шум)': A.Compose([
            A.GaussNoise(std_range=(0.01, 0.04), p=1.0)
        ]),

        'Все трансформации': A.Compose([
            A.Affine(
                scale=(0.95, 1.05),
                translate_percent=(0.03, 0.03),
                rotate=(-3, 3),
                p=0.5,
                border_mode=cv2.BORDER_CONSTANT,
                fill=0,
                fill_mask=0
            ),
            A.ElasticTransform(
                alpha=0.5,
                sigma=25,
                p=0.1,
                mask_interpolation=cv2.INTER_NEAREST
            ),
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
            A.RandomBrightnessContrast(
                brightness_limit=0.08,
                contrast_limit=0.08,
                p=0.5
            ),
            A.CoarseDropout(
                num_holes_range=(1, 2),
                hole_height_range=(0.02, 0.08),
                hole_width_range=(0.02, 0.08),
                fill=0,
                fill_mask=0,
                p=0.05
            ),
            A.GaussNoise(std_range=(0.01, 0.04), p=0.2),
        ])
    }

    return transforms_dict


def get_visualization_transforms(img_size=IMG_SIZE):
    """Трансформации без нормализации для визуализации"""
    return A.Compose([
        # Геометрические
        A.Affine(
            scale=(0.95, 1.05),
            translate_percent=(0.03, 0.03),
            rotate=(-3, 3),
            p=0.5,
            border_mode=cv2.BORDER_CONSTANT,
            fill=0,
            fill_mask=0
        ),

        # Очень мягкие деформации
        A.ElasticTransform(
            alpha=0.5,
            sigma=25,
            p=0.1,
            mask_interpolation=cv2.INTER_NEAREST
        ),

        # Специфичные для рентгена
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.5),
        A.RandomBrightnessContrast(
            brightness_limit=0.08,
            contrast_limit=0.08,
            p=0.5
        ),

        # Артефакты
        A.CoarseDropout(
            num_holes_range=(1, 2),
            hole_height_range=(0.02, 0.08),
            hole_width_range=(0.02, 0.08),
            fill=0,
            fill_mask=0,
            p=0.05
        ),

        A.CoarseDropout(
            num_holes_range=(1, 2),
            hole_height_range=(0.02, 0.08),
            hole_width_range=(0.02, 0.08),
            fill=128,
            fill_mask=0,
            p=0.05
        ),

        A.CoarseDropout(
            num_holes_range=(1, 2),
            hole_height_range=(0.02, 0.08),
            hole_width_range=(0.02, 0.08),
            fill=255,
            fill_mask=0,
            p=0.05
        ),

        # Шум
        A.GaussNoise(std_range=(0.01, 0.04), p=0.2),
    ])


def visualize_augmentations(image_path, dataset_dir='/content/dataset', show_individual=True):
    """
    Визуализирует оригинальное изображение и его трансформации

    Args:
        image_path: путь к изображению
        dataset_dir: корневая директория датасета
        show_individual: показывать отдельные трансформации (True) или случайные комбинации (False)
    """
    # Загрузка изображения
    if Path(image_path).is_absolute():
        img_full_path = image_path
    else:
        img_full_path = Path(dataset_dir) / image_path

    # Читаем изображение в градациях серого
    image = cv2.imread(str(img_full_path), cv2.IMREAD_GRAYSCALE)

    if image is None:
        raise ValueError(f"Не удалось загрузить изображение: {img_full_path}")

    if show_individual:
        # Показываем каждую трансформацию отдельно
        transforms_dict = get_individual_transforms()

        # Создаём отдельную фигуру для каждой трансформации
        for transform_name, transform in transforms_dict.items():
            # Применяем трансформацию
            transformed = transform(image=image)
            transformed_image = transformed['image']

            # Создаём фигуру с двумя изображениями рядом
            fig, axes = plt.subplots(1, 2, figsize=(14, 7))

            # Оригинальное изображение слева
            axes[0].imshow(image, cmap='gray')
            axes[0].set_title('Оригинальное изображение', fontsize=14)
            axes[0].axis('off')

            # Трансформированное изображение справа
            axes[1].imshow(transformed_image, cmap='gray')
            axes[1].set_title(transform_name, fontsize=14)
            axes[1].axis('off')

            plt.tight_layout()
            plt.show()
    else:
        # Показываем случайные комбинации всех трансформаций
        transform = get_visualization_transforms()
        num_examples = 4

        for idx in range(num_examples):
            # Применяем трансформацию
            transformed = transform(image=image)
            transformed_image = transformed['image']

            # Создаём фигуру с двумя изображениями рядом
            fig, axes = plt.subplots(1, 2, figsize=(14, 7))

            # Оригинальное изображение слева
            axes[0].imshow(image, cmap='gray')
            axes[0].set_title('Оригинальное изображение', fontsize=14)
            axes[0].axis('off')

            # Трансформированное изображение справа
            axes[1].imshow(transformed_image, cmap='gray')
            axes[1].set_title(f'Случайная комбинация #{idx + 1}', fontsize=14)
            axes[1].axis('off')

            plt.tight_layout()
            plt.show()


def visualize_random_samples(dataset_dir='/content/dataset', split='train', num_samples=2, show_individual=True):
    """
    Визуализирует случайные изображения из датасета с трансформациями

    Args:
        dataset_dir: корневая директория датасета
        split: подвыборка ('train', 'valid', 'test')
        num_samples: количество случайных изображений
        show_individual: показывать отдельные трансформации (True) или случайные комбинации (False)
    """
    images_dir = Path(dataset_dir) / split / 'images'
    image_files = list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png'))

    if not image_files:
        raise ValueError(f"Не найдено изображений в {images_dir}")

    # Выбираем случайные изображения
    selected_images = random.sample(image_files, min(num_samples, len(image_files)))

    for img_path in selected_images:
        print(f"\n{'='*80}")
        print(f"Изображение: {img_path.name}")
        print(f"{'='*80}\n")
        visualize_augmentations(
            image_path=str(img_path),
            dataset_dir='',
            show_individual=show_individual
        )

## Визуализация трансформаций

In [10]:
# Показываем каждую трансформацию отдельно
visualize_random_samples(
        dataset_dir=DATASET_DIR,
        split='train',
        num_samples=1,
        show_individual=True
    )

Output hidden; open in https://colab.research.google.com to view.

In [11]:
# Показываем набор случайных комбинаций трансформаций для изображения
visualize_random_samples(
        dataset_dir=DATASET_DIR,
        split='train',
        num_samples=1,
        show_individual=False
    )

Output hidden; open in https://colab.research.google.com to view.

# Выводы

Решено выбрать следующие варианты альбументаций для рентгеновских черно-белых снимков (ортопантомограмм, ОПТГ) из библиотеки *Albumentations*:
- геометрические преобразования с использованием `Affine` для имитаций небольших поворотов головы пациента, разного масштаба съёмки и смещения челюсти в кадре
- мягкие эластичные деформации с использованием `ElasticTransform`, которые моделирует незначительные неоднородности тканей или естественные искривления челюсти
- изменения яркости и контраста с использованием `CLAHE` (улучшает локальный контраст, делая более чёткими границы зубов и корней), `RandomBrightnessContrast` - имитирует вариации экспозиции при съёмке — разные аппараты, разные астройки яркости, толщина мягких тканей
- имитация артефактов с использованием `CoarseDropout`:

  В контексте ортопантомограмм это имитирует:
  - артефакты движения (смазанные участки)
  - наложение посторонних предметов (зажимы, маркеры)
  - отсутствие части зуба (например, разрушенная коронка)
  - переэкспонированные участки, пломбы, коронки, металлические вкладки

  Маленькая вероятность и небольшие размеры артефактов (2–8%) предотвращают
  чрезмерное зашумление датасета и позволяют модели научиться игнорировать
  локальные дефекты

- добавление шума `GaussNoise` - добавляет гауссов шум, имитируя зернистость рентгеновской плёнки или электронные шумы